# Phase 3: Feature Engineering
In this phase, we aim to extract new predictive variables from the raw dataset. [cite_start]According to our project outline, we must rely only on observable physical characteristics to predict a building's energy performance[cite: 26]. 

Our main tasks are:
1. **Deduplication:** Removing multiple EPC records for the same dwelling to prevent data leakage[cite: 35].
2. **Feature Creation:** Calculating `feature_energy_density` and `feature_building_age` proxies.

*Note: Data transformation steps that learn from the data (like scaling) will be strictly handled inside a Cross-Validation Pipeline during Phase 4 to avoid data leakage[cite: 53].*

In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset
# We load the raw dataset directly to preserve columns like BUILDING_REFERENCE_NUMBER and INSPECTION_DATE
df_raw = pd.read_parquet('manchester_epc_prepped.parquet')

print(f"Initial raw dataset size: {len(df_raw)} rows")

Initial raw dataset size: 334270 rows


## 1. Deduplication (Leakage Prevention)
To avoid target leakage caused by updated certificates for the same property, we inspect the dataset for duplicate entries and retain only the most recent record per dwelling[cite: 35].

In [2]:
# Convert inspection date to datetime objects for accurate sorting
df_raw['INSPECTION_DATE'] = pd.to_datetime(df_raw['INSPECTION_DATE'])

# Sort by inspection date and keep the latest certificate for each unique building
df_feat = df_raw.sort_values('INSPECTION_DATE').drop_duplicates(subset=['BUILDING_REFERENCE_NUMBER'], keep='last').copy()

print(f"Dataset size after dropping duplicates: {len(df_feat)} rows")
# We expect this to drop the dataset size to approximately 150,000 records.

Dataset size after dropping duplicates: 266470 rows


## 2. Feature Creation: Age Proxy
We derive a new predictive variable from the raw data. We use `CONSTRUCTION_AGE_BAND` to estimate `feature_building_age`.

**Step 2 fix:** `feature_energy_density` has been removed — it was derived from `ENERGY_CONSUMPTION_CURRENT` (an outcome variable), making it a leakage-prone feature.

In [3]:
# --- 2.1 Building Age Proxy ---
# 'construction_year_estimated' is already calculated in data preparation
# We simply subtract it from the inspection year.
inspection_year = df_feat['INSPECTION_DATE'].dt.year
df_feat['feature_building_age'] = inspection_year - df_feat['construction_year_estimated']

# Prevent negative ages (can happen due to data entry errors or specific edge cases)
df_feat['feature_building_age'] = df_feat['feature_building_age'].apply(lambda x: max(0, x) if pd.notnull(x) else x)

# Display the newly created feature
display(df_feat[['INSPECTION_DATE', 'construction_year_estimated', 'feature_building_age']].head())

,INSPECTION_DATE,construction_year_estimated,feature_building_age
161183,2006-07-20,1958.0,48.0
114244,2006-07-20,1958.0,48.0
1175,2006-07-20,1958.0,48.0
242751,2006-07-20,1958.0,48.0
223616,2007-10-11,2004.5,2.5


## 3. Final Cleanup & Export (Preventing Data Leakage)
To build a robust model that strictly relies on physical building characteristics, we must remove identification columns and variables that cause target leakage. 

Specifically, we drop:
* `ENERGY_CONSUMPTION_CURRENT` (If the model knows the exact energy consumption, predicting efficiency is trivial - severe data leakage).
* `CURRENT_ENERGY_RATING` (Already converted to binary `target_is_efficient`).
* `CURRENT_ENERGY_EFFICIENCY` (**Step 1 fix**: direct EPC outcome score — keeping it would cause leakage since it is computed from the same SAP calculation as the rating).
* ID and Date columns.

In [4]:
# Drop columns that cause leakage or are just non-predictive IDs
cols_to_drop_final = [
    'BUILDING_REFERENCE_NUMBER', 
    'INSPECTION_DATE', 
    'construction_year_estimated', # We already extracted 'feature_building_age' from this
    'CURRENT_ENERGY_RATING',       # Replaced by target_is_efficient
    'CURRENT_ENERGY_EFFICIENCY',   # Step 1: Direct EPC outcome — target leakage
    'ENERGY_CONSUMPTION_CURRENT'   # Massive target leakage variable
]

df_final = df_feat.drop(columns=[col for col in cols_to_drop_final if col in df_feat.columns])

print("=== PHASE 3 FINAL DATASET ===")
print(f"Final shape ready for Phase 4: {df_final.shape}")

# Save the final engineered dataset for Phase 4 (Modeling)
output_filename = 'manchester_epc_phase3_final.parquet'
df_final.to_parquet(output_filename)
print(f"\nSUCCESS: Phase 3 is complete! Clean and engineered data saved as '{output_filename}'.")

=== PHASE 3 FINAL DATASET ===
Final shape ready for Phase 4: (266470, 68)



SUCCESS: Phase 3 is complete! Clean and engineered data saved as 'manchester_epc_phase3_final.parquet'.
